In [1]:
import torch as t
import torch.nn as nn

In [ ]:
class RPE(nn.Module):
    def __init__(self, heads: int, seq_len: int, max_distance: int, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # shape = (distance buckets, number of heads)
        self.rpe_bias_table = nn.Embedding(max_distance+1, heads)

        positions = t.arange(seq_len)
        distances = positions.unsqueeze(1) - positions.unsqueeze(0) # shape = (L, L)
        
        c_dist = t.clamp(distances, min=0, max=max_distance) # automatically makes the negative become zero. 
        
        self.register_buffer("c_dist_cache", c_dist)

    def forward(self, x):
        
        cur_len = x.size(-1)
        c_dist = self.c_dist_cache[:cur_len, :cur_len]

        b: t.Tensor = self.rpe_bias_table(c_dist)  # dim: (L, L, n)

        rpe_bias = b.permute(2, 0, 1).unsqueeze(0) # dim: (1, n, L, L)

        return x + rpe_bias

        

In [ ]:
rpe = RPE(8, 1024, 8)
input = t.randn(4, 8, int(1024/8), int(1024/8))
output = rpe(input)

RuntimeError: The size of tensor a (128) must match the size of tensor b (1024) at non-singleton dimension 2

In [ ]:
positions = t.arange(6)

rpe_bias_table = nn.Embedding(4, 2)

distances = positions.unsqueeze(1) - positions.unsqueeze(0)
c_dist = t.clamp(distances, min=0, max=3)

b = rpe_bias_table(c_dist)

In [4]:
c_dist

tensor([[0, 0, 0, 0, 0, 0],
        [1, 0, 0, 0, 0, 0],
        [2, 1, 0, 0, 0, 0],
        [3, 2, 1, 0, 0, 0],
        [3, 3, 2, 1, 0, 0],
        [3, 3, 3, 2, 1, 0]])

In [5]:
b.permute(2, 0, 1)

tensor([[[ 1.2800,  1.2800,  1.2800,  1.2800,  1.2800,  1.2800],
         [ 0.2833,  1.2800,  1.2800,  1.2800,  1.2800,  1.2800],
         [-0.0099,  0.2833,  1.2800,  1.2800,  1.2800,  1.2800],
         [ 0.0361, -0.0099,  0.2833,  1.2800,  1.2800,  1.2800],
         [ 0.0361,  0.0361, -0.0099,  0.2833,  1.2800,  1.2800],
         [ 0.0361,  0.0361,  0.0361, -0.0099,  0.2833,  1.2800]],

        [[-0.8226, -0.8226, -0.8226, -0.8226, -0.8226, -0.8226],
         [ 2.5207, -0.8226, -0.8226, -0.8226, -0.8226, -0.8226],
         [ 1.4945,  2.5207, -0.8226, -0.8226, -0.8226, -0.8226],
         [-0.4692,  1.4945,  2.5207, -0.8226, -0.8226, -0.8226],
         [-0.4692, -0.4692,  1.4945,  2.5207, -0.8226, -0.8226],
         [-0.4692, -0.4692, -0.4692,  1.4945,  2.5207, -0.8226]]],
       grad_fn=<PermuteBackward0>)

In [6]:
b.permute(2, 0, 1).unsqueeze(0).shape

torch.Size([1, 2, 6, 6])